In [1]:
import json
import os
import numpy as np

from qutip import basis, tensor, ket2dm, ptrace, fidelity


def main():
    os.makedirs("results", exist_ok=True)

    # Single-qubit basis states
    zero = basis(2, 0)
    one = basis(2, 1)

    # Bell state |Phi+>
    phi = (tensor(zero, zero) + tensor(one, one)).unit()
    bell_proj = ket2dm(phi)

    # Initial 4-qubit state: |Phi+>_01 tensor |Phi+>_23
    psi = tensor(phi, phi)

    # Project q1,q2 onto |Phi+>
    P12 = (
        tensor(ket2dm(zero), bell_proj, ket2dm(zero))
        + tensor(ket2dm(zero), bell_proj, ket2dm(one))
        + tensor(ket2dm(one), bell_proj, ket2dm(zero))
        + tensor(ket2dm(one), bell_proj, ket2dm(one))
    )

    psi_post_unnorm = P12 * psi
    overlap = psi_post_unnorm.dag() * psi_post_unnorm
    success_probability = float(np.real(overlap))

    if success_probability < 1e-15:
        raise RuntimeError("Postselected branch has zero probability.")

    psi_post = psi_post_unnorm.unit()

    # Keep q0 and q3
    rho_03 = ptrace(ket2dm(psi_post), [0, 3])

    bell_fidelity = float(fidelity(rho_03, ket2dm(phi)) ** 2)
    entanglement_yield = success_probability * bell_fidelity

    result = {
        "stack": "qutip",
        "paradigm": "DV",
        "protocol": "two_bellpair_entanglement_swapping",
        "success_probability": success_probability,
        "bell_fidelity": bell_fidelity,
        "entanglement_yield": entanglement_yield,
        "notes": "QuTiP reference with analytic postselection on |Phi+>_12."
    }

    with open("results/qutip.json", "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2)

    print(json.dumps(result, indent=2))


if __name__ == "__main__":
    main()

{
  "stack": "qutip",
  "paradigm": "DV",
  "protocol": "two_bellpair_entanglement_swapping",
  "success_probability": 0.24999999999999978,
  "bell_fidelity": 1.0,
  "entanglement_yield": 0.24999999999999978,
  "notes": "QuTiP reference with analytic postselection on |Phi+>_12."
}
